# NLTK

In [63]:
# nltk.download('punkt_tab')
# nltk.download('averaged_perceptron_tagger_eng')
# nltk.download('maxent_ne_chunker_tab')
# nltk.download('words')
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('gutenberg')
# nltk.download('reuters')
# nltk.download('omw-1.4')

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score


import re
import nltk
from nltk.text import Text
from nltk.corpus import words, wordnet as wn, stopwords, gutenberg, reuters
from nltk import word_tokenize, sent_tokenize, ne_chunk, pos_tag, trigrams
from nltk.tokenize import WordPunctTokenizer, TreebankWordTokenizer, RegexpTokenizer
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.metrics.distance import jaccard_distance
from nltk.util import ngrams, bigrams
from nltk.probability import FreqDist
from nltk.wsd import lesk

from collections import defaultdict

In [7]:
text= "The quick brown foxes are jumping over the l@zy dog's. The sun is shining which is a fantastic sight!"

words=text.split() #Also tokenizing the text from spaces or a specific character

## Tokenization

In [73]:
# Tokenization
# downloaded punkt_tab for this
print("Original Sentence: ", text)
word_tokenized=word_tokenize(text)
print("After word tokenizing: ",word_tokenized)
sent_tokenized=sent_tokenize(text)
print("After sentence tokenizing: ",sent_tokenized)

Original Sentence:  The quick brown foxes are jumping over the l@zy dog's. The sun is shining which is a fantastic sight!
After word tokenizing:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'l', '@', 'zy', 'dog', "'s", '.', 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight', '!']
After sentence tokenizing:  ["The quick brown foxes are jumping over the l@zy dog's.", 'The sun is shining which is a fantastic sight!']


In [74]:
tokenizer= WordPunctTokenizer()
print("Original Sentence: ", text)
tokenized_text=tokenizer.tokenize(text)
print("After word punct tokenizing: ",tokenized_text)

Original Sentence:  The quick brown foxes are jumping over the l@zy dog's. The sun is shining which is a fantastic sight!
After word punct tokenizing:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'l', '@', 'zy', 'dog', "'", 's', '.', 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight', '!']


In [75]:
tokenizer=TreebankWordTokenizer()
print("Original Sentence: ", text)
tokenized_text=tokenizer.tokenize(text)
print("After word punct tokenizing: ",tokenized_text)

Original Sentence:  The quick brown foxes are jumping over the l@zy dog's. The sun is shining which is a fantastic sight!
After word punct tokenizing:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'l', '@', 'zy', "dog's.", 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight', '!']


In [77]:
tokenizer= RegexpTokenizer(r'\w+')
print("Original Sentence: ", text)
tokenized_text=tokenizer.tokenize(text)
print("After word punct tokenizing: ",tokenized_text)

Original Sentence:  The quick brown foxes are jumping over the l@zy dog's. The sun is shining which is a fantastic sight!
After word punct tokenizing:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'l', 'zy', 'dog', 's', 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight']


## Stemming

In [54]:
#Stemming 
print("Original words: ", words)
porter= PorterStemmer() # stems word by word, so using loop
stemmed_words= [porter.stem(word) for word in words] #also converts to lowercase
print("After stemming: ",stemmed_words)

print(f"Example: \n 'Playing' -> {porter.stem('playing')} \n 'played' -> {porter.stem('played')} \n 'plays' -> {porter.stem('plays')} \n 'communication' -> {porter.stem('communication')} ")

Original words:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'lazy', 'dogs.', 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight.']
After stemming:  ['the', 'quick', 'brown', 'fox', 'are', 'jump', 'over', 'the', 'lazi', 'dogs.', 'the', 'sun', 'is', 'shine', 'which', 'is', 'a', 'fantast', 'sight.']
Example: 
 'Playing' -> play 
 'played' -> play 
 'plays' -> play 
 'communication' -> commun 


## Lemmetization

In [55]:
# Lemmetization

print("Original words: ", words)
wnl= WordNetLemmatizer() #slower than stemming, also use loop because it accepts single word
lemmatized_words_without_pos= [wnl.lemmatize(word) for word in words]
print("After lemmatizing without POS: ",lemmatized_words_without_pos)

lemmatized_words_with_pos= [wnl.lemmatize(word, pos='v') for word in words] #used verb for every word for simplicity, need to pass POS accordingly for each word
print("After lemmatizing with POS: ",lemmatized_words_with_pos)

print(f"Example: \n 'Playing' -> {wnl.lemmatize('playing', pos='v')} \n 'played' -> {wnl.lemmatize('played', pos='v')} \n 'plays' -> {wnl.lemmatize('plays', pos='v')} \n 'communication' -> {wnl.lemmatize('communication')} ")

Original words:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'lazy', 'dogs.', 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight.']
After lemmatizing without POS:  ['The', 'quick', 'brown', 'fox', 'are', 'jumping', 'over', 'the', 'lazy', 'dogs.', 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight.']
After lemmatizing with POS:  ['The', 'quick', 'brown', 'fox', 'be', 'jump', 'over', 'the', 'lazy', 'dogs.', 'The', 'sun', 'be', 'shin', 'which', 'be', 'a', 'fantastic', 'sight.']
Example: 
 'Playing' -> play 
 'played' -> play 
 'plays' -> play 
 'communication' -> communication 


## Stop Words

In [ ]:
# Stop Words

stop_words = set(stopwords.words('english'))
lower_words= [word.lower() for word in words] # used this because all stopwords are in lowercase. It will not accept 'The'
print("Original words: ", lower_words)
no_stop_words= [word for word in lower_words if word not in stop_words]
print("After removing stop words: ",no_stop_words)

Original words:  ['the', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'l@zy', "dog's.", 'the', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight!']
After removing stop words:  ['quick', 'brown', 'foxes', 'jumping', 'l@zy', "dog's.", 'sun', 'shining', 'fantastic', 'sight!']


In [127]:
# Custom stopwords

print(f"Default Stopwords: {len(stop_words)} words")

custom_stopwords = {'sun', 'quick'}
custom_stopwords = stop_words.union(custom_stopwords)

print(f"Total Stopwords including custom: {len(custom_stopwords)} words\n")

print("After removing standard stop words ", no_stop_words)
no_custom_stop_words= [word for word in lower_words if word not in custom_stopwords]
print("After removing custom stop words: ",no_custom_stop_words)

Default Stopwords: 198 words
Total Stopwords including custom: 200 words

After removing standard stop words  ['quick', 'brown', 'foxes', 'jumping', 'l@zy', "dog's.", 'sun', 'shining', 'fantastic', 'sight!']
After removing custom stop words:  ['brown', 'foxes', 'jumping', 'l@zy', "dog's.", 'shining', 'fantastic', 'sight!']


## Punctuation removal

In [110]:
# Punctuation Removal

print("Original words: ", words)
no_punct_words= [re.sub(r'[^\w\s]','',words) for words in words if re.sub(r'[^\w\s]','',words)]
print("After removing punctuations: ",no_punct_words)


Original words:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'l@zy', "dog's.", 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight!']
After removing punctuations:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'lzy', 'dogs', 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight']


In [113]:
tokenizer = RegexpTokenizer(r'\w+')

print("Original words: ", words)
no_punct_words= tokenizer.tokenize(text)
print("After removing punctuations: ",no_punct_words)

Original words:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'l@zy', "dog's.", 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight!']
After removing punctuations:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'l', 'zy', 'dog', 's', 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight']


## POS Tagging

In [56]:
# Part Of Speech (POS) tagging
# Downloaded averaged_perceptron_tagger_eng for this
print("Original words: ", words)
pos_tagged_words= pos_tag(words)
print("After POS tagging: ",pos_tagged_words)


Original words:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'lazy', 'dogs.', 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight.']
After POS tagging:  [('The', 'DT'), ('quick', 'JJ'), ('brown', 'NN'), ('foxes', 'NNS'), ('are', 'VBP'), ('jumping', 'VBG'), ('over', 'IN'), ('the', 'DT'), ('lazy', 'JJ'), ('dogs.', 'NN'), ('The', 'DT'), ('sun', 'NN'), ('is', 'VBZ'), ('shining', 'VBG'), ('which', 'WDT'), ('is', 'VBZ'), ('a', 'DT'), ('fantastic', 'JJ'), ('sight.', 'NN')]


## NER

In [59]:
# Named Entity Recognition (NER)
#downloaded 'maxent_ne_chunker_tab' and 'words' for this
words1= words+['Narendra']+['Rahul']
pos_tagged_words= pos_tag(words1)
print("Original words: ", words1)
ne_chunked_words= ne_chunk(pos_tagged_words)
print("After NER: ",ne_chunked_words)

Original words:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'lazy', 'dogs.', 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight.', 'Narendra', 'Rahul']
After NER:  (S
  The/DT
  quick/JJ
  brown/NN
  foxes/NNS
  are/VBP
  jumping/VBG
  over/IN
  the/DT
  lazy/JJ
  dogs./NN
  The/DT
  sun/NN
  is/VBZ
  shining/VBG
  which/WDT
  is/VBZ
  a/DT
  fantastic/JJ
  sight./NN
  (PERSON Narendra/NNP Rahul/NNP))


## Concordance

In [85]:
conc_text=Text(word_tokenize(text))
conc_text.concordance("jumping")

Displaying 1 of 1 matches:
          The quick brown foxes are jumping over the l @ zy dog 's . The sun is


## Correcting Words

In [98]:
# doownloaded words for this.
incorrect_words=['happpye', 'azmaingi', 'intelliengxt']
correct_words=words.words()

for word in incorrect_words:
    temp = [(jaccard_distance(set(ngrams(word, 2)), set(ngrams(w, 2))),w) for w in correct_words if w[0]==word[0]]
    print(sorted(temp, key = lambda val:val[0])[0][1])

happy
amazing
intelligent


## Corpuses

In [135]:
print(gutenberg.fileids()) #prints all the available literatures available in the gutenberg corpus
bible=gutenberg.words('bible-kjv.txt')
print(bible[:20])

['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'bible-kjv.txt', 'blake-poems.txt', 'bryant-stories.txt', 'burgess-busterbrown.txt', 'carroll-alice.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'melville-moby_dick.txt', 'milton-paradise.txt', 'shakespeare-caesar.txt', 'shakespeare-hamlet.txt', 'shakespeare-macbeth.txt', 'whitman-leaves.txt']
['[', 'The', 'King', 'James', 'Bible', ']', 'The', 'Old', 'Testament', 'of', 'the', 'King', 'James', 'Bible', 'The', 'First', 'Book', 'of', 'Moses', ':']


In [153]:
synonyms= wordnet.synsets('good') # provides similar words and sometimes the same word with multiple meanings
print(synonyms)
print(synonyms[0].lemmas())
print(synonyms[1].lemmas()[0].antonyms())
print(synonyms[0].definition())
print(synonyms[0].examples())

[Synset('good.n.01'), Synset('good.n.02'), Synset('good.n.03'), Synset('commodity.n.01'), Synset('good.a.01'), Synset('full.s.04'), Synset('good.a.03'), Synset('estimable.s.01'), Synset('beneficial.s.01'), Synset('good.s.04'), Synset('good.s.05'), Synset('adept.s.01'), Synset('good.s.07'), Synset('dear.s.02'), Synset('dependable.s.03'), Synset('good.s.10'), Synset('good.s.11'), Synset('effective.s.03'), Synset('good.s.13'), Synset('good.s.14'), Synset('good.s.15'), Synset('good.s.16'), Synset('good.s.17'), Synset('good.s.18'), Synset('good.s.19'), Synset('well.r.01'), Synset('thoroughly.r.02')]
[Lemma('good.n.01.good')]
[Lemma('evil.n.03.evil')]
benefit
['for your own good', "what's the good of worrying?"]


In [157]:
sample = gutenberg.words('bible-kjv.txt')
fdist = FreqDist(sample)
print(fdist.most_common(10))

[(',', 70509), ('the', 62103), (':', 43766), ('and', 38847), ('of', 34480), ('.', 26160), ('to', 13396), ('And', 12846), ('that', 12576), ('in', 12331)]


## N-gram language modelling

In [9]:
# Creating Bigrams

bigram= bigrams(words)
for i,j in bigram:
    print(i,j)

The quick
quick brown
brown foxes
foxes are
are jumping
jumping over
over the
the l@zy
l@zy dog's.
dog's. The
The sun
sun is
is shining
shining which
which is
is a
a fantastic
fantastic sight!


In [ ]:
#predicting next word

words = nltk.word_tokenize(' '.join(reuters.words()))
tri_grams = trigrams(words)

model = defaultdict(lambda: defaultdict(lambda: 0))
for w1, w2, w3 in tri_grams:
    model[(w1, w2)][w3] += 1
    
for w1_w2 in model:
    total_count = float(sum(model[w1_w2].values()))
    for w3 in model[w1_w2]:
        model[w1_w2][w3] /= total_count
        
def predict_next_word(w1, w2):
    next_word_probs = model[w1, w2]
    if next_word_probs:
        return max(next_word_probs, key=next_word_probs.get)
    else:
        return "No prediction available"

In [201]:
word1= input("Enter the first word: ")
word2= input("Enter the second word: ")
print(f"Predicting next word:\n {word1} {word2}->{predict_next_word(word1, word2)}")

Predicting next word:
 prices increase->commodity


## Basic Sentiment Analysis

In [50]:
df= pd.read_csv(r'D:\AppStoneLab\NLP\Datasets\training.1600000.processed.noemoticon.csv.zip', encoding='latin-1', names=['target', 'ids', 'date', 'flag', 'user', 'text'])
df.head()

,target,ids,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [20]:
tweet= df['text'][0]
tokens= word_tokenize(tweet)
tagged= pos_tag(tokens)
print(tweet)
print(tokens)
print(tagged)

@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer.  You shoulda got David Carr of Third Day to do it. ;D
['@', 'switchfoot', 'http', ':', '//twitpic.com/2y1zl', '-', 'Awww', ',', 'that', "'s", 'a', 'bummer', '.', 'You', 'shoulda', 'got', 'David', 'Carr', 'of', 'Third', 'Day', 'to', 'do', 'it', '.', ';', 'D']
[('@', 'JJ'), ('switchfoot', 'NN'), ('http', 'NN'), (':', ':'), ('//twitpic.com/2y1zl', 'JJ'), ('-', ':'), ('Awww', 'NN'), (',', ','), ('that', 'WDT'), ("'s", 'VBZ'), ('a', 'DT'), ('bummer', 'NN'), ('.', '.'), ('You', 'PRP'), ('shoulda', 'VBP'), ('got', 'VBD'), ('David', 'NNP'), ('Carr', 'NNP'), ('of', 'IN'), ('Third', 'NNP'), ('Day', 'NNP'), ('to', 'TO'), ('do', 'VB'), ('it', 'PRP'), ('.', '.'), (';', ':'), ('D', 'NNP')]


In [23]:
sentence = "He went to the bank to deposit money."
tokens = word_tokenize(sentence)
sense = lesk(tokens, 'bank')
print("Best sense:", sense)
print("Definition:", sense.definition())

Best sense: Synset('depository_financial_institution.n.01')
Definition: a financial institution that accepts deposits and channels the money into lending activities


In [31]:
dog = wn.synsets('good', pos=wn.ADJ)[0]
bad = wn.synsets('bad', pos=wn.ADJ)[0]

similarity = dog.wup_similarity(bad)
print(f"Semantic Similarity (Wu-Palmer): {similarity}")

Semantic Similarity (Wu-Palmer): 0.5


In [33]:
tree = ne_chunk(tagged)
print(tree)

(S
  @/JJ
  switchfoot/NN
  http/NN
  :/:
  //twitpic.com/2y1zl/JJ
  -/:
  (GPE Awww/NN)
  ,/,
  that/WDT
  's/VBZ
  a/DT
  bummer/NN
  ./.
  You/PRP
  shoulda/VBP
  got/VBD
  (PERSON David/NNP Carr/NNP)
  of/IN
  (PERSON Third/NNP)
  Day/NNP
  to/TO
  do/VB
  it/PRP
  ./.
  ;/:
  D/NNP)


In [46]:
def semantic_analyze(text):
    tokenized= word_tokenize(text)
    tagged= pos_tag(tokenized)
    entity_chunks= ne_chunk(tagged)
    # print("Original:", text)
    # print("Tokens:", tokenized)
    # print("Named Entity tree:", entity_chunks)
    
    for word in tokenized:
        synsets= wn.synsets(word)
        if synsets:
            print(f"\nWord: {word}")
            for syn in synsets:
                print(f"    ->{syn.name()}: {syn.definition()}")
    print("\n")

In [47]:
semantic_analyze(df['text'][2])


Word: I
    ->iodine.n.01: a nonmetallic element belonging to the halogens; used especially in medicine and photography and in dyes; occurs naturally only in combination in small quantities (as in sea water or rocks)
    ->one.n.01: the smallest whole number or a numeral representing this number
    ->i.n.03: the 9th letter of the Roman alphabet
    ->one.s.01: used of a single unit or thing; not two or more

Word: dived
    ->dive.v.01: drop steeply
    ->dive.v.02: plunge into water
    ->dive.v.03: swim under water

Word: many
    ->many.a.01: a quantifier that can be used with count nouns and is often preceded by `as' or `too' or `so' or `that'; amounting to a large but indefinite number

Word: times
    ->times.n.01: a more or less definite period of time now or previously present
    ->multiplication.n.03: an arithmetic operation that is the inverse of division; the product of two numbers is computed
    ->time.n.01: an instance or single occasion for some event
    ->time.n.02:

## Text Classification

In [69]:
#Text Classification

df= pd.read_csv(r'D:\AppStoneLab\NLP\Datasets\training.1600000.processed.noemoticon.csv.zip', encoding='latin-1', names=['target', 'ids', 'date', 'flag', 'user', 'text'])
df['target'].value_counts()

target
0    800000
4    800000
Name: count, dtype: int64

In [62]:
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word.isalpha() and word not in stop_words]
    return " ".join(tokens)

df['cleaned text']= df['text'].apply(preprocess)
df.drop(['ids', 'date', 'flag', 'user', 'text'], axis=1, inplace=True)
df.head()

,target,cleaned text
0,0,switchfoot http awww bummer shoulda got david ...
1,0,upset ca update facebook texting might cry res...
2,0,kenichan dived many times ball managed save re...
3,0,whole body feels itchy like fire
4,0,nationwideclass behaving mad ca see


In [67]:
vectorizer=TfidfVectorizer()
X= vectorizer.fit_transform(df['cleaned text'])
y= df['target'].map({0:'negative',2:'neutral', 4:'positive'})


In [70]:
X_train, X_test, y_train, y_test= train_test_split(X, y, test_size=0.25, random_state=100)

model= MultinomialNB()
model.fit(X_train, y_train)
y_pred= model.predict(X_test)
cr= classification_report(y_test, y_pred)
print("Classification Report:\n",cr)
accuracy= accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

Classification Report:
               precision    recall  f1-score   support

    negative       0.75      0.79      0.77    200237
    positive       0.78      0.74      0.76    199763

    accuracy                           0.76    400000
   macro avg       0.76      0.76      0.76    400000
weighted avg       0.76      0.76      0.76    400000

Accuracy: 0.7644575


In [110]:
def predict_sentiment(text):
    text= preprocess(text)
    vectorized_text= vectorizer.transform([text])
    sentiment= model.predict(vectorized_text)[0]
    return sentiment

text=input("Enter text: ")
sentiment= predict_sentiment(text)

print(f"{text} -> Sentiment: {sentiment}")

this should return a positive output -> Sentiment: positive


# Embedding

In [65]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

from collections import Counter

from nltk.tokenize import word_tokenize

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

In [68]:
import gc

gc.collect()
torch.cuda.empty_cache()
device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Using Device:",device)
print(f"Allocated: {torch.cuda.memory_allocated(device) / (1024 ** 2):.2f} MB")
print(f"Cached: {torch.cuda.memory_reserved(device) / (1024 ** 2):.2f} MB")

Using Device: cuda
Allocated: 17.31 MB
Cached: 22.00 MB


## CBOW (Continuous Bag of Words)

In [108]:
corpus = ['The cat sat on the mat',
          'The dog ran in the park',
          'The bird sang in the tree']

corpus = [word_tokenize(sentence.lower()) for sentence in corpus]
print(corpus)

def word2idx(corpus):
    word2idx = {}
    for sentence in corpus:
        for word in sentence:
            if word not in word2idx:
                word2idx[word] = len(word2idx)+1
    return word2idx

word2idx = word2idx(corpus)
print(word2idx)

sequences=[[word2idx[word] for word in sentence] for sentence in corpus]
print(sequences)

[['the', 'cat', 'sat', 'on', 'the', 'mat'], ['the', 'dog', 'ran', 'in', 'the', 'park'], ['the', 'bird', 'sang', 'in', 'the', 'tree']]
{'the': 1, 'cat': 2, 'sat': 3, 'on': 4, 'mat': 5, 'dog': 6, 'ran': 7, 'in': 8, 'park': 9, 'bird': 10, 'sang': 11, 'tree': 12}
[[1, 2, 3, 4, 1, 5], [1, 6, 7, 8, 1, 9], [1, 10, 11, 8, 1, 12]]


In [109]:
vocab_size= len(word2idx) + 1
embedding_size = 10
window_size = 2

contexts=[]
targets=[]

for sequence in sequences:
    for i in range(window_size, len(sequence)-window_size):
        context = sequence[i-window_size:i] + sequence[i+1:i+window_size+1]
        target = sequence[i]
        contexts.append(context)
        targets.append(target)
# print(contexts)
# print(targets)

In [110]:
X = np.array(contexts)
y = np.array(targets)

class CBOWDataset(Dataset):
    def __init__(self, contexts, targets):
        self.contexts=torch.tensor(contexts, dtype=torch.long)
        self.targets=torch.tensor(targets, dtype=torch.long)
    
    def __len__(self):
        return len(self.targets)
    
    def __getitem__(self, idx):
        return self.contexts[idx], self.targets[idx]

dataset= CBOWDataset(X, y)
dataloader= DataLoader(dataset, batch_size=2, shuffle=True)

In [111]:
class CBOWModel(nn.Module):
    def __init__(self, vocab_size, embedding_size):
        super(CBOWModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.fc = nn.Linear(embedding_size, vocab_size)
    
    def forward(self, inputs):
        embedded = self.embedding(inputs)
        embedded = torch.mean(embedded, dim=1)
        output = self.fc(embedded)
        return output

model = CBOWModel(vocab_size, embedding_size)
model.to(device)

CBOWModel(
  (embedding): Embedding(13, 10)
  (fc): Linear(in_features=10, out_features=13, bias=True)
)

In [112]:
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(), lr=0.001)
n_epoch=200
for epoch in range(1,n_epoch+1):
    total_loss= 0
    correct=0
    for context, target in dataloader:
        context, target = context.to(device), target.to(device)
        optimizer.zero_grad()
        output=model(context)
        loss=loss_fn(output, target)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
        correct+=(output.argmax(1)==target).sum().item()
    if epoch%20==0:
        print(f"Epoch: {epoch}\nLoss: {total_loss/len(dataloader)}\nAccuracy: {correct/len(dataloader.dataset)*100}\n----------------------")

Epoch: 20
Loss: 2.366868257522583
Accuracy: 33.33333333333333
----------------------
Epoch: 40
Loss: 2.1170926491419473
Accuracy: 33.33333333333333
----------------------
Epoch: 60
Loss: 1.8913230895996094
Accuracy: 50.0
----------------------
Epoch: 80
Loss: 1.6872553428014119
Accuracy: 66.66666666666666
----------------------
Epoch: 100
Loss: 1.5000263055165608
Accuracy: 83.33333333333334
----------------------
Epoch: 120
Loss: 1.330199917157491
Accuracy: 83.33333333333334
----------------------
Epoch: 140
Loss: 1.1784417231877644
Accuracy: 100.0
----------------------
Epoch: 160
Loss: 1.0436147054036458
Accuracy: 100.0
----------------------
Epoch: 180
Loss: 0.9234066208203634
Accuracy: 100.0
----------------------
Epoch: 200
Loss: 0.8168148001035055
Accuracy: 100.0
----------------------


## skip-gram

In [120]:
corpus = ['The cat sat on the mat',
          'The dog ran in the park',
          'The bird sang in the tree']

corpus = [word_tokenize(sentence.lower()) for sentence in corpus]
print(corpus)

def word2idx(corpus):
    word2idx = {}
    for sentence in corpus:
        for word in sentence:
            if word not in word2idx:
                word2idx[word] = len(word2idx)+1
    return word2idx

word2idx = word2idx(corpus)
print(word2idx)

sequences=[[word2idx[word] for word in sentence] for sentence in corpus]
print(sequences)

[['the', 'cat', 'sat', 'on', 'the', 'mat'], ['the', 'dog', 'ran', 'in', 'the', 'park'], ['the', 'bird', 'sang', 'in', 'the', 'tree']]
{'the': 1, 'cat': 2, 'sat': 3, 'on': 4, 'mat': 5, 'dog': 6, 'ran': 7, 'in': 8, 'park': 9, 'bird': 10, 'sang': 11, 'tree': 12}
[[1, 2, 3, 4, 1, 5], [1, 6, 7, 8, 1, 9], [1, 10, 11, 8, 1, 12]]


In [121]:
vocab_size= len(word2idx) + 1
embedding_size = 10
window_size = 2

contexts=[]
targets=[]

for sequence in sequences:
    for i in range(window_size, len(sequence)-window_size):
        context = sequence[i-window_size:i] + sequence[i+1:i+window_size+1]
        target = sequence[i]
        contexts.append(context)
        targets.append(target)

In [122]:
# using corpus and word2idx from CBOW

def generate_skipgram_data(sequences, window_size):
    contexts = []
    targets = []
    for sequence in sequences:
        for i in range(window_size, len(sequence) - window_size):
            target = sequence[i]
            context_words = sequence[i - window_size:i] + sequence[i + 1:i + window_size + 1]
            for context_word in context_words:
                contexts.append(target)
                targets.append(context_word)
    return np.array(contexts), np.array(targets)

X, y = generate_skipgram_data(sequences, window_size)

In [123]:
class SkipGramDataset(Dataset):
    def __init__(self, contexts, targets):
        self.contexts = torch.tensor(contexts, dtype=torch.long)
        self.targets = torch.tensor(targets, dtype=torch.long)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.contexts[idx], self.targets[idx]

dataset = SkipGramDataset(X, y)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)


In [124]:
class SkipGramModel(nn.Module):
    def __init__(self, vector_size, embedding_size):
        super(SkipGramModel, self).__init__()
        self.embedding = nn.Embedding(vector_size, embedding_size)
        self.fc = nn.Linear(embedding_size, vector_size)

    def forward(self, inputs):
        embedded = self.embedding(inputs) 
        output = self.fc(embedded)
        return output
model=SkipGramModel(vocab_size, embedding_size)
model.to(device)

SkipGramModel(
  (embedding): Embedding(13, 10)
  (fc): Linear(in_features=10, out_features=13, bias=True)
)

In [ ]:
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(), lr=0.01)
n_epoch=200
for epoch in range(1,n_epoch+1):
    total_loss= 0
    # correct=0
    for context, target in dataloader:
        context, target = context.to(device), target.to(device)
        optimizer.zero_grad()
        output=model(context)
        loss=loss_fn(output, target)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
        # correct+=(output.argmax(1)==target).sum().item()
    if epoch%20==0:
        print(f"Epoch: {epoch}\nLoss: {total_loss/len(dataloader)}\n----------------------")

Epoch: 20
Loss: 1.5291657050450642
----------------------
Epoch: 40
Loss: 1.4653369287649791
----------------------
Epoch: 60
Loss: 1.4596364100774128
----------------------
Epoch: 80
Loss: 1.4435964425404866
----------------------
Epoch: 100
Loss: 1.441400796175003
----------------------
Epoch: 120
Loss: 1.4452614684899647
----------------------
Epoch: 140
Loss: 1.4427416026592255
----------------------
Epoch: 160
Loss: 1.4607226848602295
----------------------
Epoch: 180
Loss: 1.4332001010576885
----------------------
Epoch: 200
Loss: 1.4400496035814285
----------------------


# Subword Tokenization

In [85]:
# imports
import re
from collections import defaultdict
from transformers import AutoTokenizer

## BPE

In [ ]:
def get_stats(vocab): 
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i + 1]] += freq
    return pairs

def merge_vocab(pair, v_in):
    v_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in v_in:
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

def get_vocab(data):
    vocab = defaultdict(int)
    for line in data:
        for word in line.split():
            vocab[' '.join(list(word)) + ' </w>'] += 1
    return vocab

def byte_pair_encoding(data, n):
    vocab = get_vocab(data)
    for i in range(n):
        pairs = get_stats(vocab)
        best = max(pairs, key=pairs.get)
        vocab = merge_vocab(best, vocab)
    return vocab

In [82]:
corpus = '''Morphemes are the smallest units of meaning in language, 
forming the foundation of words. Each morpheme, whether a root, prefix, 
or suffix, carries distinct meaning. Combined, they create the complex words
we use daily. Understanding morphemes is crucial in linguistic analysis,
breaking down words into core components and revealing patterns. This insight enhances language processing tasks, 
helping bridge the gap between raw linguistic data and meaningful interpretation in language learning, translation, 
or NLP.'''

data= corpus.split('.')

vocab=get_vocab(data)
print(vocab)
pairs=get_stats(vocab) # creates character pairs of 2 
print(pairs)
best=max(pairs, key=pairs.get) # gets most common character pairs (in this case 'i' 'n') 
print(best)
vocab=merge_vocab(best, vocab) # merges only the best character pairs, keeping rest the same ('i' 'n' -> 'in')
print(vocab)

defaultdict(<class 'int'>, {'M o r p h e m e s </w>': 1, 'a r e </w>': 1, 't h e </w>': 4, 's m a l l e s t </w>': 1, 'u n i t s </w>': 1, 'o f </w>': 2, 'm e a n i n g </w>': 2, 'i n </w>': 3, 'l a n g u a g e , </w>': 1, 'f o r m i n g </w>': 1, 'f o u n d a t i o n </w>': 1, 'w o r d s </w>': 3, 'E a c h </w>': 1, 'm o r p h e m e , </w>': 1, 'w h e t h e r </w>': 1, 'a </w>': 1, 'r o o t , </w>': 1, 'p r e f i x , </w>': 1, 'o r </w>': 2, 's u f f i x , </w>': 1, 'c a r r i e s </w>': 1, 'd i s t i n c t </w>': 1, 'C o m b i n e d , </w>': 1, 't h e y </w>': 1, 'c r e a t e </w>': 1, 'c o m p l e x </w>': 1, 'w e </w>': 1, 'u s e </w>': 1, 'd a i l y </w>': 1, 'U n d e r s t a n d i n g </w>': 1, 'm o r p h e m e s </w>': 1, 'i s </w>': 1, 'c r u c i a l </w>': 1, 'l i n g u i s t i c </w>': 2, 'a n a l y s i s , </w>': 1, 'b r e a k i n g </w>': 1, 'd o w n </w>': 1, 'i n t o </w>': 1, 'c o r e </w>': 1, 'c o m p o n e n t s </w>': 1, 'a n d </w>': 2, 'r e v e a l i n g </w>': 1, 

In [83]:
data= corpus.split('.')

vocab=byte_pair_encoding(data, 230)
print(vocab)

{'Morphemes</w>': 1, 'are</w>': 1, 'the</w>': 4, 'smallest</w>': 1, 'units</w>': 1, 'of</w>': 2, 'meaning</w>': 2, 'in</w>': 3, 'language,</w>': 1, 'forming</w>': 1, 'foundation</w>': 1, 'words</w>': 3, 'Each</w>': 1, 'morpheme,</w>': 1, 'whether</w>': 1, 'a</w>': 1, 'root,</w>': 1, 'prefix,</w>': 1, 'or</w>': 2, 'suffix,</w>': 1, 'carries</w>': 1, 'distinct</w>': 1, 'Combined,</w>': 1, 'they</w>': 1, 'create</w>': 1, 'complex</w>': 1, 'we</w>': 1, 'use</w>': 1, 'daily</w>': 1, 'Understanding</w>': 1, 'morphemes</w>': 1, 'is</w>': 1, 'crucial</w>': 1, 'linguistic</w>': 2, 'analysis,</w>': 1, 'breaking</w>': 1, 'down</w>': 1, 'into</w>': 1, 'core</w>': 1, 'components</w>': 1, 'and</w>': 2, 'revealing</w>': 1, 'patterns</w>': 1, 'This</w>': 1, 'insight</w>': 1, 'enhances</w>': 1, 'language</w>': 2, 'processing</w>': 1, 'tasks,</w>': 1, 'helping</w>': 1, 'bridge</w>': 1, 'gap</w>': 1, 'between</w>': 1, 'raw</w>': 1, 'data</w>': 1, 'meaningful</w>': 1, 'interpretation</w>': 1, 'learning,</

## WordPiece

In [86]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_wp(text):
    # text is input as sentences
    print("Original Text: ", text)

    tokens= tokenizer.tokenize(text)
    print("WordPiece Tokens:", tokens)
    
    token_ids= tokenizer.convert_tokens_to_ids(tokens)
    print("Token IDs:", token_ids)
    
    words= text.split()
    print("\nWord Breakdown:\n")
    
    for word in words:
        word_tokens= tokenizer.tokenize(word)
        print(f" {word} -> {word_tokens}")
    
corpus = '''Morphemes are the smallest units of meaning in language, 
forming the foundation of words. Each morpheme, whether a root, prefix, 
or suffix, carries distinct meaning. Combined, they create the complex words
we use daily. Understanding morphemes is crucial in linguistic analysis,
breaking down words into core components and revealing patterns. This insight enhances language processing tasks, 
helping bridge the gap between raw linguistic data and meaningful interpretation in language learning, translation, 
or NLP.'''

corpus= corpus.lower().split('.')

for sentence in corpus:
    tokenize_wp(sentence)
    print('-'*100)

Original Text:  morphemes are the smallest units of meaning in language, 
forming the foundation of words
WordPiece Tokens: ['m', '##or', '##phe', '##mes', 'are', 'the', 'smallest', 'units', 'of', 'meaning', 'in', 'language', ',', 'forming', 'the', 'foundation', 'of', 'words']
Token IDs: [182, 1766, 27801, 6801, 1132, 1103, 10471, 2338, 1104, 2764, 1107, 1846, 117, 5071, 1103, 4686, 1104, 1734]

Word Breakdown:

 morphemes -> ['m', '##or', '##phe', '##mes']
 are -> ['are']
 the -> ['the']
 smallest -> ['smallest']
 units -> ['units']
 of -> ['of']
 meaning -> ['meaning']
 in -> ['in']
 language, -> ['language', ',']
 forming -> ['forming']
 the -> ['the']
 foundation -> ['foundation']
 of -> ['of']
 words -> ['words']
----------------------------------------------------------------------------------------------------
Original Text:   each morpheme, whether a root, prefix, 
or suffix, carries distinct meaning
WordPiece Tokens: ['each', 'm', '##or', '##phe', '##me', ',', 'whether', 'a', 

# Transformer

## Self-Attention

In [21]:
import torch
from torch import nn
from torch.nn import functional as F

In [ ]:
sentence= 'The quick brown fox jumps over the lazy dog'

word2idx= {word:idx for idx, word in enumerate(sorted(sentence.split()))}
print(word2idx)

{'The': 0, 'brown': 1, 'dog': 2, 'fox': 3, 'jumps': 4, 'lazy': 5, 'over': 6, 'quick': 7, 'the': 8}


In [11]:
mapped= [word2idx[i] for i in sentence.split()]
mapped= torch.tensor(mapped)
print(mapped)

tensor([0, 7, 1, 3, 4, 6, 8, 5, 2])


In [18]:
torch.manual_seed(100)
vocab_size=50000
embed= nn.Embedding(vocab_size, 3)
embedded=embed(mapped).detach() # detach to stop gradient accumulation
print(embedded)
print(embedded.shape)

tensor([[ 0.1268,  1.3564,  0.5632],
        [-0.5721, -1.2546,  0.0486],
        [-0.1039, -0.3575,  0.3917],
        [ 1.2426,  0.5403, -1.1454],
        [-1.4592, -1.6281,  0.3834],
        [-0.0247, -0.8466,  0.0293],
        [ 1.1705, -0.5410, -0.7116],
        [-0.1718, -3.1896,  1.5914],
        [-0.6801,  0.2409,  0.4698]])
torch.Size([9, 3])


In [ ]:
embedding_dim= embedded.shape[1]
d_q, d_k, d_v = 2, 2, 4

W_query= nn.Parameter(torch.rand(embedding_dim, d_q))
W_key= nn.Parameter(torch.rand(embedding_dim, d_k))
W_value= nn.Parameter(torch.rand(embedding_dim, d_v))

Q= torch.matmul(embedded, W_query) 
K= torch.matmul(embedded, W_key)
V= torch.matmul(embedded, W_value)

torch.Size([9, 2])
torch.Size([9, 2])
torch.Size([9, 4])


In [24]:
attention_score= torch.matmul(Q, K.T) / (d_q**0.5)
attention_weight= F.softmax(attention_score, dim=-1)

print(attention_score)
print(attention_weight)

tensor([[ 1.0751, -0.4666,  0.3394, -0.8078, -0.4648, -0.1967, -0.5983,  1.0050,
          0.3957],
        [-0.9511,  0.4362, -0.2898,  0.6552,  0.4697,  0.1780,  0.4822, -0.8494,
         -0.3216],
        [-0.1647,  0.0887, -0.0444,  0.0801,  0.1142,  0.0330,  0.0571, -0.1248,
         -0.0397],
        [ 0.3452, -0.2074,  0.0833, -0.1131, -0.2933, -0.0729, -0.0761,  0.2250,
          0.0570],
        [-1.3090,  0.6180, -0.3910,  0.8568,  0.6907,  0.2479,  0.6280, -1.1390,
         -0.4211],
        [-0.5650,  0.2559, -0.1736,  0.3974,  0.2710,  0.1052,  0.2929, -0.5101,
         -0.1950],
        [-0.2870,  0.0931, -0.1046,  0.2957,  0.0453,  0.0472,  0.2232, -0.3218,
         -0.1440],
        [-1.7790,  0.8578, -0.5234,  1.1189,  0.9836,  0.3400,  0.8173, -1.5175,
         -0.5505],
        [ 0.1316, -0.0363,  0.0508, -0.1517, -0.0048, -0.0206, -0.1151,  0.1583,
          0.0737]], grad_fn=<DivBackward0>)
tensor([[0.2521, 0.0539, 0.1208, 0.0384, 0.0540, 0.0707, 0.0473, 0.2350, 0.

In [25]:
context_vector= torch.matmul(attention_weight, V)
print(context_vector)
print(context_vector.shape)

tensor([[-0.2907, -0.3611,  0.4224,  0.0106],
        [-0.5202, -0.2571, -0.1045, -0.5795],
        [-0.4629, -0.3407,  0.0565, -0.3845],
        [-0.3472, -0.2730,  0.1955, -0.2135],
        [-0.5525, -0.2599, -0.1564, -0.6424],
        [-0.4907, -0.2855, -0.0352, -0.4937],
        [-0.4355, -0.2462,  0.0275, -0.4179],
        [-0.5869, -0.2697, -0.2084, -0.7040],
        [-0.4325, -0.3682,  0.1379, -0.2932]], grad_fn=<MmBackward0>)
torch.Size([9, 4])


## Hugging Face Library Basic

In [46]:
import torch
from transformers import pipeline
from transformers import AutoTokenizer
from transformers import AutoModel
from transformers import AutoModelForSequenceClassification

In [2]:
classifier = pipeline("sentiment-analysis")
classifier(
    [
        "i love this so much",
        "I hate this so much!",
    ]
)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 740.58it/s, Materializing param=pre_classifier.weight]                                  


[{'label': 'POSITIVE', 'score': 0.9998810291290283},
 {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

In [3]:
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
raw_inputs = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt") # used in future cells
print(inputs)

{'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102],
        [  101,  1045,  5223,  2023,  2061,  2172,   999,   102,     0,     0,
             0,     0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])}


In [4]:
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModel.from_pretrained(checkpoint)
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 725.00it/s, Materializing param=transformer.layer.5.sa_layer_norm.weight]   
DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
pre_classifier.weight | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([2, 16, 768])


In [14]:
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
outputs = model(**inputs)
print(outputs.logits.shape)
print(outputs.logits)
predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(predictions)
model.config.id2label

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 841.10it/s, Materializing param=pre_classifier.weight]                                  


torch.Size([2, 2])
tensor([[-1.5607,  1.6123],
        [ 4.1692, -3.3464]], grad_fn=<AddmmBackward0>)
tensor([[4.0195e-02, 9.5980e-01],
        [9.9946e-01, 5.4418e-04]], grad_fn=<SoftmaxBackward0>)


{0: 'NEGATIVE', 1: 'POSITIVE'}

In [15]:
model = AutoModel.from_pretrained("bert-base-cased")
# or
# from transformers import BertModel
# model = BertModel.from_pretrained("bert-base-cased")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 769.39it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
model.save_pretrained("bert_model")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.29it/s]


In [ ]:
model = AutoModel.from_pretrained("bert_model")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 636.62it/s, Materializing param=pooler.dense.weight]                               


In [17]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

encoded_input = tokenizer("Hello, I'm a single sentence!")
print(encoded_input)
tokenizer.decode(encoded_input["input_ids"])

{'input_ids': [101, 8667, 117, 146, 112, 182, 170, 1423, 5650, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


"[CLS] Hello, I ' m a single sentence! [SEP]"

In [18]:
encoded_input = tokenizer("How are you?", "I'm fine, thank you!")
print(encoded_input)

{'input_ids': [101, 1731, 1132, 1128, 136, 102, 146, 112, 182, 2503, 117, 6243, 1128, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [19]:
encoded_input = tokenizer("How are you?", "I'm fine, thank you!", return_tensors="pt")
print(encoded_input)

encoded_input = tokenizer("How are you?", "I'm fine, thank you!", return_tensors="pt", padding=True, truncation=True, max_length=5)
print(encoded_input)

{'input_ids': tensor([[ 101, 1731, 1132, 1128,  136,  102,  146,  112,  182, 2503,  117, 6243,
         1128,  106,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
{'input_ids': tensor([[ 101, 1731,  102,  146,  102]]), 'token_type_ids': tensor([[0, 0, 0, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}


In [59]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

sequence = "Using a Transformer Samuel network is simple"
tokens = tokenizer.tokenize(sequence)

print(tokens)

['Using', 'a', 'Trans', '##former', 'Samuel', 'network', 'is', 'simple']


In [21]:
ids = tokenizer.convert_tokens_to_ids(tokens)

print(ids)

[7993, 170, 13809, 23763, 2443, 1110, 3014]


In [22]:
decoded_string = tokenizer.decode([7993, 170, 11303, 1200, 2443, 1110, 3014])
print(decoded_string)

Using a transformer network is simple


In [23]:
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence = "I've been waiting for a HuggingFace course my whole life."

tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)
input_ids = torch.tensor(ids)
# This line will fail.
model(input_ids) # intentional error, to check for dimension requirements

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 810.64it/s, Materializing param=pre_classifier.weight]                                  


RuntimeError: The size of tensor a (14) must match the size of tensor b (512) at non-singleton dimension 1

In [24]:
tokenized_inputs = tokenizer(sequence, return_tensors="pt")
print(tokenized_inputs["input_ids"])

tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102]])


In [25]:
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence = "I've been waiting for a HuggingFace course my whole life."

tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)

input_ids = torch.tensor([ids])
print("Input IDs:", input_ids)

output = model(input_ids)
print("Logits:", output.logits)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 842.08it/s, Materializing param=pre_classifier.weight]                                  

Input IDs: tensor([[ 1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,  2607,
          2026,  2878,  2166,  1012]])
Logits: tensor([[-2.7276,  2.8789]], grad_fn=<AddmmBackward0>)


In [26]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence1_ids = [[200, 200, 200]]
sequence2_ids = [[200, 200]]
batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer.pad_token_id], # to maintain same size 
]

print(model(torch.tensor(sequence1_ids)).logits)
print(model(torch.tensor(sequence2_ids)).logits)
print(model(torch.tensor(batched_ids)).logits)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 788.32it/s, Materializing param=pre_classifier.weight]                                  

tensor([[ 1.5694, -1.3895]], grad_fn=<AddmmBackward0>)
tensor([[ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)
tensor([[ 1.5694, -1.3895],
        [ 1.3374, -1.2163]], grad_fn=<AddmmBackward0>)


In [27]:
batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer.pad_token_id],
]

attention_mask = [
    [1, 1, 1],
    [1, 1, 0], # 0 notifies to ignore the padded token, hence output will be similar to sequence2_ids
]

outputs = model(torch.tensor(batched_ids), attention_mask=torch.tensor(attention_mask))
print(outputs.logits)

tensor([[ 1.5694, -1.3895],
        [ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

sequences = ["I've been waiting for a HuggingFace course my whole life.", "So have I!"]

model_inputs_og = tokenizer(sequences)
print(model_inputs_og['input_ids'])

model_inputs = tokenizer(sequences, padding="longest") # longest sentence
print(model_inputs['input_ids'])

model_inputs = tokenizer(sequences, padding="max_length") # max length supported by model (512 here)
print(model_inputs['input_ids'])

model_inputs = tokenizer(sequences, padding="max_length", max_length=8) # max length set to 8
print(model_inputs['input_ids'])

model_inputs = tokenizer(sequences, truncation=True) # truncate only the ones that are longer than model limit (512 here)
print(model_inputs['input_ids'])

model_inputs = tokenizer(sequences, max_length=8, truncation=True) # truncate after 8
print(model_inputs['input_ids'])

tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens) # this will remove 2 special words that are added by default in tokenizer by default, one from start and one from end
print(ids)

print(tokenizer.decode(model_inputs_og["input_ids"])) # this will have special tokens
print(tokenizer.decode(ids)) # this will not have special tokens

[[101, 146, 112, 1396, 1151, 2613, 1111, 170, 20164, 10932, 2271, 7954, 1736, 1139, 2006, 1297, 119, 102], [101, 1573, 1138, 146, 106, 102]]
[[101, 146, 112, 1396, 1151, 2613, 1111, 170, 20164, 10932, 2271, 7954, 1736, 1139, 2006, 1297, 119, 102], [101, 1573, 1138, 146, 106, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
[[101, 146, 112, 1396, 1151, 2613, 1111, 170, 20164, 10932, 2271, 7954, 1736, 1139, 2006, 1297, 119, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [29]:
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
sequences = ["I've been waiting for a HuggingFace course my whole life.", "So have I!"]

tokens = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")
output = model(**tokens)

print(output.logits)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 624.56it/s, Materializing param=pre_classifier.weight]                                  


tensor([[-1.5607,  1.6123],
        [-3.6183,  3.9137]], grad_fn=<AddmmBackward0>)


## Fine-Tuning

In [1]:
import torch
from torch.optim import AdamW
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel, DataCollatorWithPadding
from transformers import AutoModelForMaskedLM
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import get_scheduler
from tqdm.auto import tqdm
import evaluate

C:\Users\Lenovo\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Pre-processing for fine-tuning

In [21]:
#Example of training

checkpoint= "bert-base-uncased"
tokenizer= AutoTokenizer.from_pretrained(checkpoint)
model= AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequences = [
    "I've been waiting for a HuggingFace course my whole life.",
    "This course is amazing!",
]
batch= tokenizer(sequences, padding= True, Truncation= True, return_tensors= "pt")

batch["labels"]=torch.tensor([1,1])
optimizer=AdamW(model.parameters())
loss= model(**batch).loss
loss.backward()
optimizer.step()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 675.00it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those pa

In [22]:
raw_datasets= load_dataset("glue", "mrpc")
raw_train_dataset = raw_datasets["train"]
print(raw_train_dataset[0])
raw_train_dataset.features

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .', 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .', 'label': 1, 'idx': 0}


{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

In [ ]:
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True) # returns in "[CLS] sentence 1 tokens [SEP] sentence 2 tokens [SEP]" format

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True) # map is the efficient way to handle tokenization of entire dataset at once as tokenizer expects only a single input
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

In [26]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [27]:
samples = tokenized_datasets["train"][:8]
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}
[len(x) for x in samples["input_ids"]]

[50, 59, 47, 67, 59, 50, 62, 32]

In [ ]:
batch = data_collator(samples) # padding is done based on the maximum length in the batch
{k: v.shape for k, v in batch.items()}

{'input_ids': torch.Size([8, 67]),
 'token_type_ids': torch.Size([8, 67]),
 'attention_mask': torch.Size([8, 67]),
 'labels': torch.Size([8])}

### BERT

In [74]:
raw_dataset= load_dataset("glue", "mrpc")
model_name= "bert-base-uncased"
tokenizer= AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_dataset.map(
    tokenize_function, 
    batched=True, 
    remove_columns=["sentence1", "sentence2", "idx"] # Drop only the strings/metadata
)
data_collator= DataCollatorWithPadding(tokenizer= tokenizer)
tokenized_datasets

Map: 100%|██████████| 1725/1725 [00:00<00:00, 17288.72 examples/s]


DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

In [75]:
tokenized_datasets = tokenized_datasets.rename_column("label", "labels") # renamed just for convenience of model
tokenized_datasets.set_format("torch")
print(tokenized_datasets["train"][0].keys())

dict_keys(['labels', 'input_ids', 'token_type_ids', 'attention_mask'])


In [76]:
train_dataloader = DataLoader(tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator, num_workers=4, pin_memory=True)
eval_dataloader = DataLoader(tokenized_datasets["validation"], batch_size=8, collate_fn=data_collator, num_workers=4, pin_memory=True)


In [77]:
#just to check if the padding works correctly by chceking size
for batch in train_dataloader: 
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([8]),
 'input_ids': torch.Size([8, 64]),
 'token_type_ids': torch.Size([8, 64]),
 'attention_mask': torch.Size([8, 64])}

In [78]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 702.15it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those pa

In [79]:
outputs = model(**batch) 
print(outputs.loss, outputs.logits.shape) # just to test working

tensor(0.8797, grad_fn=<NllLossBackward0>) torch.Size([8, 2])


In [80]:
optimizer = AdamW(model.parameters(), lr=5e-5)

num_epoch= 3
num_training_steps= num_epoch * len(train_dataloader)
lr_scheduler= get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)
print(num_training_steps)

1377


In [81]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
device

device(type='cuda')

In [ ]:
# progress_bar = tqdm(range(num_training_steps))
num_epoch= 3
model.train()
for epoch in range(num_epoch):
    progress_bar = tqdm(range(len(train_dataloader)), desc=f"Epoch {epoch}")
    for batch in train_dataloader:
        batch= {k: v.to(device) for k, v in batch.items()}
        outputs= model(**batch)
        loss= outputs.loss
        
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)
        progress_bar.set_postfix({"loss": loss.item()})

Epoch 2: 100%|██████████| 459/459 [03:20<00:00,  2.36it/s, loss=0.00921]

In [87]:
metric = evaluate.load("glue", "mrpc")
model.eval()
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch["labels"])

metric.compute()

{'accuracy': 0.8504901960784313, 'f1': 0.8950086058519794}

In [88]:
model.save_pretrained("./mrpc-bert")
tokenizer.save_pretrained("./mrpc-bert")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.07it/s]


('./mrpc-bert\\tokenizer_config.json', './mrpc-bert\\tokenizer.json')

In [91]:
model_path= "./mrpc-bert"
model= AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer= AutoTokenizer.from_pretrained(model_path)

model.to(device)
model.eval()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 341.38it/s, Materializing param=classifier.weight]


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [130]:
sentence1 = "Nitish killed a Lion."
sentence2 = "Lion died by natural casues."

inputs = tokenizer(
    sentence1,
    sentence2,
    return_tensors="pt",
    padding=True,
    truncation=True,
)

inputs = {k: v.to(device) for k, v in inputs.items()}

In [137]:
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

pred = torch.argmax(logits, dim=1).item()

label_map = {0: "not paraphrase", 1: "paraphrase"}
print("Prediction:", label_map[pred])

probs = torch.softmax(logits, dim=1)
print("Probabilities:", probs.tolist())

Prediction: not paraphrase
Probabilities: [[0.9758960008621216, 0.024104034528136253]]


### GPT-2

In [141]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

In [142]:
model_name= "gpt2"
tokenizer= GPT2Tokenizer.from_pretrained(model_name)
model= GPT2LMHeadModel.from_pretrained(model_name, pad_token_id= tokenizer.eos_token_id)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 319.73it/s, Materializing param=transformer.wte.weight]
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
sequence= "What is AI?"

inputs=tokenizer.encode(sequence, return_tensors= "pt")
outputs= model.generate(inputs, max_length=20, num_beams=3, early_stopping= True, do_sample= True, no_repeat_ngram_size= 3, temperature= 1)

In [154]:
text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(text)

What is AI?

AI is an artificial intelligence system that learns from the human mind. It's a machine learning system that has the ability to learn from human memories. It has a lot of potential.

What is the future of AI? What is the potential of AI and how do you think about it? What do you see as the future? How do we think about AI? Let's talk about the future. Let's start with the past. There's a lot going on in the world right now. We've all been here for a long time. We're all here to do a job. We have jobs to do. And we all want to do it. We want to be able to do things that we can't do in the real world. We don't have to do all the things we can do. We can do things we could never do in our own lives. And that's the future we're talking about. That's where we're going.


## Pipeline Inference

In [1]:
import transformers
from transformers import pipeline
from transformers import AutoTokenizer
from transformers import AutoModelForTokenClassification

C:\Users\Lenovo\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Text Generation

In [21]:
generator= pipeline("text-generation", model= "Qwen/Qwen3-0.6B")
# generator.generation_config
generator("Waking up early is beneficial for", early_stopping= True, max_new_tokens= 50)

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 783.52it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "Waking up early is beneficial for your health and happiness, and it's important to have a good sleep. The main reason why some people wake up early is that they have a good sleep. So the correct answer is A.\nA. Correct\nB. Incorrect\n\nThe main reason"}]

In [15]:
model_path = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_path)
generator = pipeline(
    "text-generation",
    model=model_path,
    tokenizer=tokenizer,
    device_map="cuda"
)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain the concept of LLM in 90 words."}
]

output = generator(
    messages,
    max_new_tokens=100,      
    do_sample=True,
    early_stopping= True,          
    temperature=0.3,         
    top_p=0.9,               
    top_k=5,               
    eos_token_id=[151645, 151643] # Qwen specific EOS token IDs
)

print(output)

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 422.41it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'Explain the concept of LLM in 90 words.'}, {'role': 'assistant', 'content': "<think>\nOkay, the user wants me to explain the concept of LLM in 90 words. Let me start by recalling what I know about LLMs. They're large language models, right? So I need to define them clearly.\n\nFirst, I should mention that LLMs are artificial intelligence models trained on vast amounts of text data. Then, their ability to understand and generate human-like text. Maybe include examples like answering questions or writing articles. Also, note that they can learn"}]}]


### NER

In [34]:
model_name= "dslim/bert-base-NER"
tokenizer= AutoTokenizer.from_pretrained(model_name)
model= AutoModelForTokenClassification.from_pretrained(model_name)

ner_pipeline= pipeline(task="ner", model= model, tokenizer= tokenizer)

text= "Donald Trump is the president of USA."

result=ner_pipeline(text)
print("\n")
for dict in result:
    print(f"word: {dict['word']}, entity: {dict['entity']}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 773.85it/s, Materializing param=classifier.weight]                                      
BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.




word: Donald, entity: B-PER
word: Trump, entity: I-PER
word: USA, entity: B-LOC


In [88]:
text1= "I plan to go on a world tour starting from either Sydney or New York and it would cost roughly around $100,000 which I will borrow from the Bank."
ner_pipeline1= pipeline(task= "ner", aggregation_strategy= "simple")
result1=ner_pipeline1(text1)
for dict in result1:
    print(f"word: {dict['word']}, entity: {dict['entity_group']}")

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 697.85it/s, Materializing param=classifier.weight]                                      
BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


word: Sydney, entity: LOC
word: New York, entity: LOC
word: Bank, entity: ORG


### Mask-filling

In [24]:
unmasker= pipeline("fill-mask", model= "FacebookAI/xlm-roberta-base")
unmasker("The best food in the world is <mask>.") # the masking symbol is differnet for each model

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 624.42it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
XLMRobertaForMaskedLM LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'score': 0.058469533920288086,
  'token': 143896,
  'token_str': 'chicken',
  'sequence': 'The best food in the world is chicken .'},
 {'score': 0.04510254040360451,
  'token': 67155,
  'token_str': 'fish',
  'sequence': 'The best food in the world is fish .'},
 {'score': 0.041163887828588486,
  'token': 64507,
  'token_str': 'chocolate',
  'sequence': 'The best food in the world is chocolate .'},
 {'score': 0.03306122496724129,
  'token': 96967,
  'token_str': 'cheese',
  'sequence': 'The best food in the world is cheese .'},
 {'score': 0.03290897235274315,
  'token': 147836,
  'token_str': 'vegetarian',
  'sequence': 'The best food in the world is vegetarian .'}]

In [32]:
text= "This is a <mask> of a sentence."
result= unmasker(text, top_k=3)

for dict in result:
    print(f"word: {dict['token_str']}, score: {dict['score']}")
    print(dict['sequence'])
    print('-'*100)

word: fragment, score: 0.11662566661834717
This is a fragment of a sentence.
----------------------------------------------------------------------------------------------------
word: summary, score: 0.08322945982217789
This is a summary of a sentence.
----------------------------------------------------------------------------------------------------
word: definition, score: 0.06290211528539658
This is a definition of a sentence.
----------------------------------------------------------------------------------------------------


### Q&A

In [37]:
qa= pipeline("question-answering")

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 102/102 [00:00<00:00, 758.35it/s, Materializing param=qa_outputs.weight]                                     


In [41]:
question="What is the projected year for ROI, and how does it compare to the original projection?"
context=r''' 
Project Nebula: Annual Technical Review 2025
Executive Summary: Project Nebula, initialized in January 2023, aimed to modernize the regional power distribution system in the Aurora District by deploying advanced AI-driven transformers. The core objective was to replace aging infrastructure (1980s-era conventional transformers) with new Type-X4 smart transformers to reduce energy loss by 15% and increase grid uptime.
Phase 1 (Jan 2023-Dec 2023): Focus was on upgrading the North substation. The team replaced 50 units. During this phase, early data indicated a 7% efficiency gain, lower than expected, due to integration issues with the existing legacy SCADA systems.
Phase 2 (Jan 2024-June 2024): The team addressed the SCADA issue by implementing the "QuantumLink" API bridging software. Following this, the South substation was upgraded, adding 75 new Type-X4 units.
Current Status (as of Oct 2025): The total number of deployed units has reached 180, including 55 units recently installed in the East Industrial Sector. The overall system efficiency gain, measured across all three sectors (North, South, East), is now at 16.5%.
Technical Data & Challenges: The X4 units have an operating temperature range of -20°C to +55°C. A persistent issue in the North Sector resulted in 12 units overheating during the summer of 2024, necessitating the installation of secondary cooling units (Model-SC1) on 8 of those, while 4 were completely replaced.
Future Outlook: The final phase (West Sector) will begin in 2026. Data suggests that the return on investment (ROI) will be achieved by 2028, two years ahead of the original, more conservative 2030 projection.
'''
qa(question= question, context= context)

{'score': 0.1772187054157257,
 'start': 1595,
 'end': 1650,
 'answer': 'two years ahead of the original, more conservative 2030'}

### Zero shot 

In [48]:
zs= pipeline("zero-shot-classification", model= "facebook/bart-large-mnli")

Loading weights: 100%|██████████| 515/515 [00:02<00:00, 218.43it/s, Materializing param=model.shared.weight]                                   


In [58]:
sequence= "Michael jordan is a president contender after his basketball season"
labels= ["technology", "sports", "politics", "cooking"]

zs(sequence, labels)

{'sequence': 'Michael jordan is a president contender after his basketball season',
 'labels': ['sports', 'politics', 'technology', 'cooking'],
 'scores': [0.804140031337738,
  0.19016224145889282,
  0.0034874307457357645,
  0.0022103083319962025]}